# Runbook 5.1.a: SQL injection  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Mitigate Risk · **Maps to:** 5.1.A, 5.1.B · **Language:** Python 3 (pytest, sqlite3)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** 

**Lesson objective.** Identify SQL injection attack patterns in web application input, explain how unsanitized queries expose database contents, and apply detection logic to flag malicious parameter strings.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def authenticate(users: list[tuple[str, str, str]], username: str, password: str) -> str | None
```

#### Input
- `users`: a list of `(username, role, password)` rows to load into an in-memory table.
- `username`, `password`: the submitted login credentials (untrusted input).

#### Output
- The matching user's `role` (a string) when a row's username AND password both equal the
  submitted values.
- `None` when no row matches, including every injection attempt.

#### Guarantees / rules
- Deterministic, no network. Standard library only (`sqlite3`).
- The username/password comparison must be performed **by a SQL query**, using a
  parameterized statement with `?` placeholders and the values bound separately.
- No input filtering, blocklisting, or Python-side list comparison.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).
the query is parameterized rather than built by string concatenation.

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
import sqlite3


def authenticate(users, username, password):
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("CREATE TABLE users (username TEXT, role TEXT, password TEXT)")
    cur.executemany("INSERT INTO users VALUES (?, ?, ?)", users)

    cur.execute(
        "SELECT role FROM users WHERE username = ? AND password = ?",
        (username, password),
    )
    row = cur.fetchone()
    conn.close()
    return row[0] if row else None


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

USERS = [
    ("admin", "administrator", "S3cr3t-Admin-Pw"),
    ("jordan", "staff", "autumn-maple-31"),
    ("priya", "staff", "river-stone-77"),
]


def test_valid_admin():
    assert impl.authenticate(USERS, "admin", "S3cr3t-Admin-Pw") == "administrator"


def test_valid_staff():
    assert impl.authenticate(USERS, "jordan", "autumn-maple-31") == "staff"


def test_wrong_password_rejected():
    assert impl.authenticate(USERS, "admin", "not-the-password") is None


def test_unknown_user_rejected():
    assert impl.authenticate(USERS, "nobody", "whatever") is None


def test_comment_truncation_blocked():
    assert impl.authenticate(USERS, "admin'--", "anything") is None


def test_or_bypass_blocked():
    assert impl.authenticate(USERS, "' OR '1'='1", "x") is None


def test_or_bypass_with_comment_blocked():
    assert impl.authenticate(USERS, "' OR '1'='1'--", "x") is None


def test_union_blocked():
    assert impl.authenticate(USERS, "' UNION SELECT role FROM users--", "x") is None


def test_injection_in_password_blocked():
    assert impl.authenticate(USERS, "admin", "' OR '1'='1") is None


def test_literal_quote_username():
    users = USERS + [("o'brien", "staff", "pw-123")]
    assert impl.authenticate(users, "o'brien", "pw-123") == "staff"
    assert impl.authenticate(users, "o'brien", "wrong") is None


def test_empty_users():
    assert impl.authenticate([], "admin", "x") is None


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it